# 04 — Controlled AAD experiments
Lists the fixed controlled AAD plan and optionally runs exactly one guarded stage. Diagnostic dataset selection does not affect this notebook.

In [1]:
from pathlib import Path
import subprocess
import sys
REPO_ROOT = Path('/content/phd_research')
if not REPO_ROOT.is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/sanelehlabisa/phd_research.git', str(REPO_ROOT)], check=True)
else:
    checkout_dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip()
    if checkout_dirty:
        print('Existing Colab checkout has local changes; preserving them and skipping update.')
    else:
        subprocess.run(['git', 'pull', '--ff-only', 'origin', 'master'], cwd=REPO_ROOT, check=True)
IMPLEMENTATION_ROOT = REPO_ROOT / 'papers/001-journal-abnormal-activity-recognition/implementation'
bootstrap = subprocess.run([sys.executable, str(IMPLEMENTATION_ROOT / 'src/colab_bootstrap.py'), str(IMPLEMENTATION_ROOT / 'requirements.txt')])
if bootstrap.returncode == 75:
    raise SystemExit('Restart the Colab runtime, reconnect to the A100, and rerun from the top.')
bootstrap.check_returncode()
if str(IMPLEMENTATION_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPLEMENTATION_ROOT))
import importlib
for module_name in [name for name in sys.modules if name == 'src' or name.startswith('src.')]:
    del sys.modules[module_name]
importlib.invalidate_caches()


In [2]:
from src.notebook_utils import controlled_plan_list_command, controlled_stage_command, run_command, validate_runtime
validate_runtime(require_cuda=True)
run_command(controlled_plan_list_command(IMPLEMENTATION_ROOT), IMPLEMENTATION_ROOT)


In [3]:
SELECTED_STAGE = 'architecture-screen'
CANDIDATE_EXPANSION_DECISION = 'pending'  # pending, skip, or expanded
RUN_CONTROLLED_EXPERIMENT = False
print({'stage': SELECTED_STAGE, 'candidate_expansion': CANDIDATE_EXPANSION_DECISION, 'enabled': RUN_CONTROLLED_EXPERIMENT})


{'stage': 'architecture-screen', 'candidate_expansion': 'pending', 'enabled': False}


In [4]:
if RUN_CONTROLLED_EXPERIMENT:
    command = controlled_stage_command(IMPLEMENTATION_ROOT, SELECTED_STAGE, CANDIDATE_EXPANSION_DECISION)
    run_command(command, IMPLEMENTATION_ROOT)
else:
    print('Controlled execution disabled; plan listing only.')


Controlled execution disabled; plan listing only.


## Validation winner and longer confirmation
After the complete architecture screen, paste its exact run directory. The winner is read from the saved validation ranking, never typed manually.

In [ ]:
from src.notebook_utils import confirmation_stage_command, validation_screen_winner
SCREEN_RUN_DIR = ''
RUN_LONGER_CONFIRMATION = True
if SCREEN_RUN_DIR:
    screen_winner = validation_screen_winner(SCREEN_RUN_DIR)
    print({'validation_winner': screen_winner['name'], 'validation_metrics': screen_winner['validation_metrics']})
else:
    print('Set SCREEN_RUN_DIR after the complete validation screen.')


ImportError: cannot import name 'confirmation_stage_command' from 'src.notebook_utils' (/content/phd_research/papers/001-journal-abnormal-activity-recognition/implementation/src/notebook_utils.py)

In [ ]:
if RUN_LONGER_CONFIRMATION:
    confirmation_command = confirmation_stage_command(IMPLEMENTATION_ROOT, SCREEN_RUN_DIR, CANDIDATE_EXPANSION_DECISION)
    run_command(confirmation_command, IMPLEMENTATION_ROOT)
else:
    print('Longer confirmation disabled.')


## One-time final test evaluation
Only enable this after both confirmation seeds finish. The helper validates the screen winner, confirmation provenance, checkpoint, and split before opening test data.

In [ ]:
from src.notebook_utils import final_test_evaluation_command, frozen_confirmation_checkpoint
CONFIRMATION_RUN_DIRS = []
FINAL_TEST_ACKNOWLEDGEMENT = ''  # Set exactly: OPEN FINAL AAD TEST
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    frozen_checkpoint = frozen_confirmation_checkpoint(SCREEN_RUN_DIR, CONFIRMATION_RUN_DIRS)
    final_command = final_test_evaluation_command(IMPLEMENTATION_ROOT, frozen_checkpoint, FINAL_TEST_ACKNOWLEDGEMENT)
    run_command(final_command, IMPLEMENTATION_ROOT)
else:
    print('Final test remains locked.')


In [ ]:
from src.notebook_utils import final_test_report, printable
FINAL_EVALUATION_RUN_DIR = ''
if FINAL_EVALUATION_RUN_DIR:
    final_report = final_test_report(FINAL_EVALUATION_RUN_DIR)
    print(printable(final_report))
    from IPython.display import Video, display
    for example in final_report['examples']:
        display(Video(example['path'], embed=True, width=360))
else:
    print('Set FINAL_EVALUATION_RUN_DIR after the one-time evaluation to display its five test predictions.')
